## Week 2 Day 2 - オーケストレーション

いよいよ最初のAgenticフレームワークプロジェクトです！！

### パート1: メールのセットアップ

### パート2: コードによるオーケストレーション

### パート3: LLMによるオーケストレーション

- 3a: Tools経由
- 3b: Handoffs経由

## パート1: メールのセットアップ

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">重要 必ずお読みください - メールの送信について</h2>
            <span style="color:#ff7800;">これから、メールを送信するエージェントを作成します。これを行う最良の方法は、SendGridやResendのようなメールプロバイダーを使うことです。しかし、これにはかなりの手間がかかります。自分がドメインを所有している正式なメールホストから送信する必要があり、それをDNSレコードで証明しなければなりません。かなり面倒です。<br/><br/>
            そこで、無料でシンプルな方法を使います。SMTPサーバーの設定を使って、自分のメールアドレスから直接送信するのです。設定は簡単ですが、その分機能は限定的です（例えばメールを受信することはできません）。もっと本格的に進めたい場合は、代わりにSendGridやResendを使ってください。<br/><br/>
            そして実は、メール送信自体は必須ではありません。ここでは、Agentがメールを送信する様子を示すためにこれを行っています。重要なのはAgentの働きであり、メール送信そのものではありません。お好みで、この関数をPushoverのプッシュ通知に置き換えても構いません。
            </span>
        </td>
    </tr>
</table>

## 自分のSMTPサーバーからメールを送信するための設定

### ステップ1: SMTPサーバーを確認する

Googleで検索するか、ChatGPTやClaudeに自分のメールアドレスのSMTPサーバーを聞いてみましょう。よく使われるものを以下にいくつか挙げます。一部のメールプロバイダーではSMTPサーバーが有効になっていない場合があります（例：仕事や学校向けのMicrosoft 365）。

Google: smtp.gmail.com  
Outlook.com / Hotmail / Live: smtp-mail.outlook.com  
Microsoft 365: smtp.office365.com  
iCloud Mail: smtp.mail.me.com  

.envファイルに以下を追加してください。

`EMAIL_SMTP_SERVER=xxxx`

### ステップ2: アプリ専用パスワードを取得する

自分のメールプロバイダーでの取得方法をGoogleで検索してください。Gmailの場合、2段階認証を有効にしておく必要があります。そのうえで、次のページにアクセスします。

https://myaccount.google.com/apppasswords

任意の名前を付け、パスワードをコピーして.envファイルに追加します。その際、付与されるスペースは取り除いてください（16文字、スペースなしになるはずです）。

`EMAIL_APP_PASSWORD=xxxx`

### ステップ3: 自分のメールアドレスを追加する

`EMAIL_ADDRESS=xxx`

.envファイルを保存するのを忘れずに！

In [ ]:
from dotenv import load_dotenv
import requests
from agents import Agent, Runner, trace, function_tool, ModelSettings
from agents.extensions.visualization import draw_graph
from openai.types.responses import ResponseTextDeltaEvent
import os
import asyncio
import smtplib
from email.message import EmailMessage
load_dotenv(override=True)
MODEL_NAME = "gpt-5.4-mini"

In [ ]:
EMAIL_ADDRESS = os.getenv("EMAIL_ADDRESS")
EMAIL_SMTP_SERVER = os.getenv("EMAIL_SMTP_SERVER")
EMAIL_APP_PASSWORD = os.getenv("EMAIL_APP_PASSWORD")

if EMAIL_ADDRESS:
    print("Email address is set")
else:
    print("Email address is not set")

if EMAIL_SMTP_SERVER:
    print("SMTP server is set")
else:
    print("SMTP server is not set")

if EMAIL_APP_PASSWORD:
    print("App password is set")
else:
    print("App password is not set")

USE_EMAIL = EMAIL_ADDRESS and EMAIL_SMTP_SERVER and EMAIL_APP_PASSWORD

if USE_EMAIL:
    print("Email is set up and we will try using it")
else:
    print("Email is not set up; we will send push notifications instead")

In [ ]:
# さあ、やってみましょう

def send_email(subject, text_body, html_body):
    msg = EmailMessage()
    msg["From"] = EMAIL_ADDRESS
    msg["To"] = EMAIL_ADDRESS
    msg["Subject"] = subject
    msg.set_content(text_body)
    msg.add_alternative(html_body, subtype="html")

    with smtplib.SMTP(EMAIL_SMTP_SERVER, 587) as server:
        server.starttls()
        server.login(EMAIL_ADDRESS, EMAIL_APP_PASSWORD)
        server.send_message(msg)

In [ ]:
send_email("Testing testing 123", "Fingers crossed..", "<html><body><strong>Fingers</strong> crossed..</body></html>")

### これがうまくいかなかった場合は、以下のコメントを外してメールを使わないようにしてください

In [ ]:
# USE_EMAIL = False

### フォールバック戦略 - プッシュ通知を送る

In [ ]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and looks good")
    else:
        print("Pushover user found but doesn't start with u")
else:
    print("Pushover user not found")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks good")
    else:
        print("Pushover token found but doesn't start with a")
else:
    print("Pushover token not found")

In [ ]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [ ]:
def send_message(subject, text_body, html_body):
    if USE_EMAIL:
        send_email(subject, text_body, html_body)
    else:
        push(f"Subject: {subject}\n\n{text_body}")

### よし、これでもう全部動くはずです！

In [ ]:
send_message("Big news", "Communications are a go!", "<html><body>Communications are a <strong>go!</strong></body></html>")

## Agentのオーケストレーション

Agentのオーケストレーションには2つのモデルがあります。コードによるものと、LLMによるものです。

コードによる方法: より予測可能で決定論的です。

LLMによる方法: より強力です。

以下に素晴らしい解説があります。

https://openai.github.io/openai-agents-python/multi_agent/

まずはコードによる方法から始めます。

## パート2: コードによるオーケストレーション

In [ ]:
intro = """
You are a sales agent working for ComplAI, 
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI.
You write emails.
"""

instructions1 = intro + "Your email style is professional, serious, with gravitas and credibility."
instructions2 = intro + "Your email style is witty, engaging, and humorous."
instructions3 = intro + "Your email style is concise, to the point, in the style of a busy senior executive."

In [ ]:
sales_agent1 = Agent(name="Professional Sales Agent", instructions=instructions1, model=MODEL_NAME)
sales_agent2 = Agent(name="Humorous Sales Agent", instructions=instructions2, model=MODEL_NAME)
sales_agent3 = Agent(name="Executive Sales Agent", instructions=instructions3, model=MODEL_NAME)


In [ ]:

result = Runner.run_streamed(sales_agent1, input="Write a cold sales email")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

In [ ]:
message = "Write a cold sales email"

with trace("Parallel cold emails"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )

outputs = [result.final_output for result in results]

for output in outputs:
    print(output + "\n\n")


In [ ]:
decision = """
You pick the best cold sales email from the given options.
Imagine you are a customer and pick the one you are most likely to respond to.
Do not give an explanation; reply with the selected email only.
"""

sales_picker = Agent(name="Sales_picker", instructions=decision, model=MODEL_NAME)


In [ ]:
message = "Write a cold sales email"

with trace("Sales selection workflow"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )
    outputs = [result.final_output for result in results]

    emails = "Cold sales emails:\n\n" + "\n\nEmail:\n\n".join(outputs)

    best = await Runner.run(sales_picker, emails)

    print(f"Best sales email:\n{best.final_output}")


さあ、traceを確認してみましょう。

https://platform.openai.com/traces

### では、ここにツールを加えてみましょう。

In [ ]:
@function_tool
def send_email_tool(subject: str, text_body: str, html_body: str) -> str:
    """
    Send out an email with the given subject and body to all sales prospects
    
    Args:
        subject: The subject of the email
        text_body: The body of the email as plain text
        html_body: The HTML body of the email
    """
    send_message(subject, text_body, html_body)
    return "Email sent successfully"

### これは自動的にツールへと変換され、定型的なjsonも生成されました

In [ ]:
send_email_tool.params_json_schema

In [ ]:
decision = """
You pick the best cold sales email from the given options.
Imagine you are a customer and pick the one you are most likely to respond to.
Then use your tool to send the email.
"""

require_tool = ModelSettings(tool_choice="required")

sales_sender = Agent(name="Sales Sender", instructions=decision, model=MODEL_NAME, tools=[send_email_tool], model_settings=require_tool)

In [ ]:
message = "Write a cold sales email"

with trace("Sales selection workflow with sending"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )
    outputs = [result.final_output for result in results]

    emails = "Cold sales emails:\n\n" + "\n\nEmail:\n\n".join(outputs)

    response = await Runner.run(sales_sender, emails)

    print(f"Final response:\n{response.final_output}")

### うまくいきましたか？！

詳しくはtraceを見てください！これはデバッグに非常に役立つ方法です。より小さなモデルでは、より多くの時間と試行が必要になるかもしれません。

https://platform.openai.com/traces

## パート3: LLMによるオーケストレーション

### 3a: Tools経由

1つのAgentが別のAgentを呼び出すことを選択する最もシンプルな方法は、それをツール呼び出しとして扱うことです。

OpenAI Agents SDKには、これを行うための非常にシンプルな方法が用意されています。

これは、次のようなフローで最もうまく機能します。

Agent A -> Agent B -> Agent A

そして、典型的な「Planning Agent」的な状況にも向いています。

In [ ]:
description = "Use this tool to write a sales email. In the input, just instruct it to write a sales email."

tool1 = sales_agent1.as_tool(tool_name="sales_email_writer_1", tool_description=description)
tool1

### では、すべてのツールをまとめてみましょう。

3つのメール作成Agentそれぞれに対するツール

そして、メールを送信する関数用のツール

In [ ]:
tool1 = sales_agent1.as_tool(tool_name="sales_email_writer_1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_email_writer_2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_email_writer_3", tool_description=description)

tools = [tool1, tool2, tool3, send_email_tool]

tools

## そしていよいよ、Sales Manager（プランニングエージェント）の登場です

In [ ]:
instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_writer tools.
"""

task = """
Follow these steps:

1. Generate Drafts: Use each of the three sales_email_writer tools to generate different email drafts.
Just instruct each to write a sales email; no further details are needed.
Do not proceed until all three drafts are ready, one from each tool.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
 
3. Use your tool to send the best email (and only the best email) to the user. Only send 1 email.
"""

sales_manager = Agent(name="Sales Manager", instructions=instructions, tools=tools, model=MODEL_NAME)


In [ ]:
draw_graph(sales_manager)

In [ ]:
with trace("Sales manager"):
    result = await Runner.run(sales_manager, task)

## traceを確認するのを忘れずに

https://platform.openai.com/traces

そして自分のメールを確認しましょう！！迷惑メール（Junk / Spam）フォルダも見てください。何しろ、これは基本的にスパムメッセージのようなものですから……


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">結果が安定しない？</h2>
            <span style="color:#ff7800;">これはAgentic AI、特にLLMによるオーケストレーションではよくあることです。
            これを解決するには、プロンプトを試行錯誤しながら反復していく必要があります。特に小さなモデルでは、信頼できる結果を得るために
            より多くの実験が必要になる場合があります。
            </span>
        </td>
    </tr>
</table>

## パート3: LLMによるオーケストレーション

### 3a: Handoffs経由

私はHandoffsがあまり好きではありません。かなり信頼性に欠けるように見えますし、他のフレームワークでも一貫して使われているわけではありません。

裏側では、OpenAI Agents SDKもこれをToolsを使って実装しています。

### Handoffsは、あるAgentが別のAgentに処理を委任し、制御を渡す方法を表します

HandoffsとAgents-as-toolsは似ています。

どちらの場合も、あるAgentが別のAgentと協調して動作できます。

ツールを使う場合、制御は戻ってきます。

A -> B -> A

Handoffsを使う場合、制御はそのまま渡ります。

A -> B

In [ ]:

instructions = """
You are a Sales Manager at ComplAI. You get your sales team to draft emails, then send them all to a sales picker.
"""

task = """
Follow these steps:

1. Generate Drafts: Use each of the three sales_email_writer tools to generate different email drafts.
Just instruct each to write a sales email; no further details are needed.
Do not proceed until all three drafts are ready, one from each tool.
 
2. Handoff to the sales sender to choose and send the best email.
"""

tools = [tool1, tool2, tool3]
handoffs = [sales_sender]

sales_manager = Agent(name="Sales Manager", instructions=instructions, tools=tools, handoffs=handoffs, model=MODEL_NAME)


In [ ]:
draw_graph(sales_manager)

In [ ]:
with trace("Sales manager"):
    result = await Runner.run(sales_manager, task)

### traceを確認するのを忘れずに

https://platform.openai.com/traces

そして自分のメールを確認しましょう！！

なお、Handoffsは信頼性が低く、少しもどかしく感じることがあります。これを機能させるには、ツールの使用を強制する必要がありました。安定した動作が得られない場合は、プロンプトを反復して調整するか、より大きなモデルを使ってみてください。そして、そのプロセス自体を楽しんでください。これこそがAgentic AIの本質なのです！

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">演習</h2>
            <span style="color:#ff7800;">さらにオーケストレーションを拡張してみましょう。例えば、メールをもう一度反復して洗練させるステップを加えるなど。
            コードによるオーケストレーションと、LLMによるオーケストレーションの両方を使ってみてください。  
            難しい課題: SendGridやResendのような本格的なメールプロバイダーを使うように切り替えてみましょう。そのうえで、ユーザーからの返信を処理します。
            さらに、SDRが応答して会話を続けられるようにしましょう！
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">商業的な意義</h2>
            <span style="color:#00bfff;">これはセールスの自動化にすぐに応用できますが、より一般的には、会話とツールを通じたあらゆるビジネスプロセスのエンドツーエンドな自動化にも応用できます。このようなAgentソリューションを、自分の日々の仕事にどう活かせるか考えてみてください。
            </span>
        </td>
    </tr>
</table>